# CRAFT Phase 2: Curriculum-guided Reinforced Adaptive Fine-Tuning (RL Loop)
This notebook trains the model using GRPO reinforcement learning incorporating:
- Component A: Symbolic Math & NLI Logical Step Verifiers
- Component B: Contrastive Step-Level Preference DPO (starts at step 100)
- Component C: Adaptive Curriculum selecting from the 40-70% difficulty zone

In [ ]:
# 1. Install dependencies
!pip install -q transformers bitsandbytes accelerate datasets peft trl loguru pyyaml scipy numpy sentence-transformers

In [ ]:
# 2. Setup Kaggle Workspace (Copy Datasets)
# Scans /kaggle/input for Phase 0 (difficulty map) and Phase 1 (SFT checkpoints) outputs
import os
import shutil

print("Searching for Phase 0 and Phase 1 data in /kaggle/input...")
diff_map = None
chk_dir = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'difficulty_map.json' in files and not diff_map:
        diff_map = os.path.join(root, 'difficulty_map.json')
    
    if 'checkpoints' in dirs and not chk_dir:
        possible_dir = os.path.join(root, 'checkpoints')
        if os.path.exists(os.path.join(possible_dir, 'sft', 'final')):
            chk_dir = possible_dir

if diff_map:
    os.makedirs("data", exist_ok=True)
    shutil.copy(diff_map, "data/difficulty_map.json")
    shutil.copy(diff_map, "difficulty_map.json")
    print("\u2705 Copied difficulty_map.json")
else:
    print("\u26a0\ufe0f Could not find difficulty_map.json! Did you attach the Phase 0 dataset?")

if chk_dir:
    if not os.path.exists("checkpoints"):
        shutil.copytree(chk_dir, "checkpoints")
    print("\u2705 Copied checkpoints/sft/final")
else:
    print("\u26a0\ufe0f Could not find checkpoints/sft/final! Did you attach the Phase 1 dataset?")

In [ ]:
# 3. Verify required files exist
import os
assert os.path.exists("checkpoints/sft/final"), "Error: checkpoints/sft/final not found. Attach Phase 1 dataset!"
assert os.path.exists("difficulty_map.json"), "Error: difficulty_map.json not found. Attach Phase 0 dataset!"

In [ ]:
# 4. Run Pre-Training Diagnostics
# Verifies that the reward scorer, curriculum engine, and KL controller are functioning correctly.
!python -m src.phase2_rl.run_diagnostics

In [ ]:
# 5. Run CRAFT RL Loop training (Start from Step 1)
# Note: Starts from scratch. No --resume flag.
!python -m src.phase2_rl.craft_rl_loop --config phi3_mini --hardware kaggle --output checkpoints/rl

In [ ]:
# 6. Verify RL checkpoints
final_path = "checkpoints/rl/final"
if os.path.exists(final_path):
    print("CRAFT RL Loop training successfully completed. Weights saved at:", final_path)
    print(os.listdir(final_path))
else:
    print("Error: checkpoints/rl/final not found!")